# Intraday Volatility Forecasting: GRU

**Author:** Ayooluwa Adelagun  
**Dataset:** NVDA Intraday Volatility Dataset

---

## Purpose

This notebook implements the GRU intraday forecasting experiment reported in **Chapter 7** of the dissertation, producing the GRU row of **Table 7.1**.

The task is to predict whether daily realised volatility will increase or decrease relative to the prior trading day. The five-feature input set is: `Volume`, `Return`, `Return Squared`, `VIX`, and `Daily Volatility`.

Hyperparameters are selected via a random search over 50 iterations with five-fold walk-forward cross-validation, optimising out-of-sample accuracy on the validation folds. The best model is then evaluated on the held-out test set spanning late 2021 through early 2026.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, GRU, Input, Activation
from tensorflow.keras.models import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit
from scipy.stats import randint
import itertools
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

The intraday dataset is loaded. The target distribution is printed as a sanity check: a near 50/50 split confirms the dataset is balanced, making accuracy a meaningful metric and removing the need for class weighting during training.

In [ ]:
data_full = pd.read_csv('NVDA_Intraday_Volatility_Dataset_VIX.csv')

print(f'Total rows: {len(data_full):,}')
print(f'Date range: {data_full["Date"].iloc[0]} to {data_full["Date"].iloc[-1]}')
print(f'Target distribution:\n{data_full["target"].value_counts(normalize=True).round(3)}')

## 3. Feature and Target Arrays

The five input features and the binary target are extracted.

In [ ]:
X = data_full[["Volume", "Return", "Return_Squared", "VIX", "Daily Volatility"]]
Y = data_full["target"]
data_set = data_full[["Date", "Volume", "Return", "Return_Squared", "VIX", "Daily Volatility", "target"]]

## 4. Train-Test Split

The dataset is split chronologically at the 80% mark, placing the cutoff in late 2021. The test set spans late 2021 through early 2026 and remains entirely unseen until the final evaluation step.

In [ ]:
# Chronological 80/20 split
splitlimit = int(len(data_full) * 0.8)
training_features = data_set[:splitlimit].copy()
test_data = data_set[splitlimit:]

print(f'Training rows: {len(training_features):,}')
print(f'Test rows:     {len(test_data):,}')

## 5. Outlier Removal

Outliers are removed from the **training set only** using a rolling median absolute deviation (MAD) approach with a centred window of 41 observations. The procedure is applied sequentially across all five features; any row flagged on any feature is removed. This mirrors the pipeline described in Section 3.7 of the dissertation.

In [ ]:
# Outlier removal — 2x MAD across all 5 features
training_features["volatility_rolling_median"] = training_features["Daily Volatility"].rolling(window=41, center=True, min_periods=1).median()
training_features["return_squared_rolling_median"] = training_features["Return_Squared"].rolling(window=41, center=True, min_periods=1).median()
training_features["return_rolling_median"] = training_features["Return"].rolling(window=41, center=True, min_periods=1).median()
training_features["VIX_rolling_median"] = training_features["VIX"].rolling(window=41, center=True, min_periods=1).median()
training_features["volume_rolling_median"] = training_features["Volume"].rolling(window=41, center=True, min_periods=1).median()

training_features["volatility minus median"] = (training_features["Daily Volatility"] - training_features["volatility_rolling_median"]).abs()
training_features["return_squared minus median"] = (training_features["Return_Squared"] - training_features["return_squared_rolling_median"]).abs()
training_features["return minus median"] = (training_features["Return"] - training_features["return_rolling_median"]).abs()
training_features["VIX minus median"] = (training_features["VIX"] - training_features["VIX_rolling_median"]).abs()
training_features["volume minus median"] = (training_features["Volume"] - training_features["volume_rolling_median"]).abs()

volatility_outliers_removed = training_features[
    ~(training_features['volatility minus median'] > 2 * training_features['volatility minus median'].median())
]
all_outliers_removed = volatility_outliers_removed[
    ~(volatility_outliers_removed['return_squared minus median'] > 2 * volatility_outliers_removed['return_squared minus median'].median())
]
all_outliers_removed = all_outliers_removed[
    ~(all_outliers_removed['return minus median'] > 2 * volatility_outliers_removed['return minus median'].median())
]
all_outliers_removed = all_outliers_removed[
    ~(all_outliers_removed['VIX minus median'] > 2 * volatility_outliers_removed['VIX minus median'].median())
]
all_outliers_removed = all_outliers_removed[
    ~(all_outliers_removed['volume minus median'] > 2 * volatility_outliers_removed['volume minus median'].median())
]

removed = len(training_features) - len(all_outliers_removed)
print(f'Rows before: {len(training_features):,}')
print(f'Rows after:  {len(all_outliers_removed):,}')
print(f'Removed:     {removed:,} ({removed / len(training_features) * 100:.1f}%)')

## 6. Cleaned Feature and Target Arrays

Feature matrices and the target vector are re-extracted from the cleaned training set.

In [ ]:
X_cleaned = all_outliers_removed[["Volume", "Return", "Return_Squared", "VIX", "Daily Volatility"]]
Y_cleaned = all_outliers_removed["target"]
data_set_cleaned = all_outliers_removed[["Volume", "Return", "Return_Squared", "VIX", "Daily Volatility", "target"]]

## 7. Feature Scaling

Min-max scaling is applied to the feature columns. The scaler is fitted exclusively on the cleaned training data.

In [ ]:
# Scale features
scaler = MinMaxScaler()
training_data_features_scaled = scaler.fit_transform(X_cleaned)
data_set_scaled = scaler.fit_transform(data_set_cleaned)

## 8. Sequence Construction (Training)

Input sequences of length `backcandles = 15` are constructed from the scaled training features. Each sequence $\mathbf{x}_{t-L+1:t} \in \mathbb{R}^{L \times 5}$ is paired with the target label at time $t+1$.

In [ ]:
# Sequence construction — 15 backcandles, 5 features
backcandles = 15
n_features  = 5
Z = []

for j in range(n_features):
    Z.append([])
    for i in range(backcandles, training_data_features_scaled.shape[0]):
        Z[j].append(training_data_features_scaled[i-backcandles:i, j])

Z = np.moveaxis(Z, [0], [2])
Z, yi = np.array(Z), np.array(data_set_scaled[backcandles-1:, -1])
y_final = np.reshape(yi, (len(yi), 1))[1:]

print(f'Training sequences shape: {Z.shape}')
print(f'Training targets shape:   {y_final.shape}')

## 9. Hyperparameter Search

A random search over 50 parameter configurations is conducted, with each configuration evaluated using five-fold walk-forward (time-series) cross-validation. The search ranges over hidden units (50 to 150), batch size (16, 32, 64), and number of epochs (10 to 30). 

Walk-forward cross-validation is used, consistent with the time-series nature of the data.

In [ ]:
# Random search with walk-forward cross-validation

def create_model(units=80):
    gru_input = Input(shape=(backcandles, n_features), name='gru_input')
    inputs = GRU(units, name='first_layer')(gru_input)
    inputs = Dense(1, name='dense_layer')(inputs)
    output = Activation('sigmoid', name='output')(inputs)
    model = Model(inputs=gru_input, outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

param_dist = {
    'units':      randint(50, 150),
    'batch_size': [16, 32, 64],
    'epochs':     randint(10, 30),
}

n_iter         = 50
best_score     = -np.inf
best_params    = None
best_model_path = 'NVDA_GRU_VIX_best_model.keras'
tscv           = TimeSeriesSplit(n_splits=5)

print(f'Running random search: {n_iter} iterations x 5 time-series folds')
print('-' * 55)

for iteration, params in enumerate(ParameterSampler(param_dist, n_iter=n_iter, random_state=42)):
    fold_scores = []

    for fold, (train_index, val_index) in enumerate(tscv.split(Z)):
        X_train_fold = Z[train_index]
        X_val_fold   = Z[val_index]
        y_train_fold = y_final[train_index]
        y_val_fold   = y_final[val_index]

        model = create_model(units=params['units'])
        model.fit(
            X_train_fold, y_train_fold,
            epochs=params['epochs'],
            batch_size=params['batch_size'],
            verbose=0
        )

        _, score = model.evaluate(X_val_fold, y_val_fold, verbose=0)
        fold_scores.append(score)

    avg_score = np.mean(fold_scores)
    print(f'Iteration {iteration+1}/{n_iter} | params: {params} | avg val accuracy: {avg_score:.4f}')

    if avg_score > best_score:
        best_score  = avg_score
        best_params = params
        model.save(best_model_path)

print('-' * 55)
print(f'Best params: {best_params}')
print(f'Best avg val accuracy: {best_score:.4f}')

## 10. Load Best Model

The best model identified during the hyperparameter search is loaded.

In [ ]:
# Load best model
from tensorflow.keras.models import load_model
best_model = load_model(best_model_path)
best_model.summary()

## 11. In-Sample Predictions

The best model generates predictions on the training sequences as a sanity check.

In [ ]:
# In-sample predictions
validation_predictions = best_model.predict(Z)
validation_predicted_classes = (validation_predictions > 0.5).astype(int)
dataframe_val = pd.DataFrame(y_final, columns=['target'])
dataframe_val['predicted'] = validation_predicted_classes
cm_insample = confusion_matrix(dataframe_val['target'], dataframe_val['predicted'])
print('In-sample Confusion Matrix:')
print(cm_insample)

## 12. Test Set Preparation

The held-out test features are extracted and scaled using the scaler fitted on the training data.

In [ ]:
# Scale test data
X_test       = test_data[["Volume", "Return", "Return_Squared", "VIX", "Daily Volatility"]]
Y_test       = test_data["target"]
test_dataset = test_data[["Volume", "Return", "Return_Squared", "VIX", "Daily Volatility", "target"]]

test_scaled   = scaler.fit_transform(test_dataset)
X_test_scaled = scaler.fit_transform(X_test)

## 13. Test Sequence Construction

Test sequences of length `backcandles = 15` are constructed from the scaled test features, mirroring the approach used for the training sequences.

In [ ]:
# Reconstruct test sequences
T = []

for j in range(n_features):
    T.append([])
    for i in range(backcandles, X_test_scaled.shape[0]):
        T[j].append(X_test_scaled[i-backcandles:i, j])

T = np.moveaxis(T, [0], [2])
T, yi_test = np.array(T), np.array(test_scaled[backcandles-1:, -1])
y_final_test = np.reshape(yi_test, (len(yi_test), 1))[1:]

print(f'Test sequences shape: {T.shape}')
print(f'Test targets shape:   {y_final_test.shape}')

## 14. Out-of-Sample Predictions

The best GRU model generates probability scores on the test sequences. A threshold of 0.5 is applied to produce binary class predictions. The confusion matrix is printed.

In [ ]:
# Out-of-sample predictions and confusion matrix
test_predictions = best_model.predict(T)
test_predicted_classes = (test_predictions > 0.5).astype(int)
dataframe = pd.DataFrame(y_final_test, columns=['target'])
dataframe['predicted'] = test_predicted_classes

y_true = dataframe['target'].values.astype(int)
cm_outsample = confusion_matrix(y_true, test_predicted_classes)
print('Out-of-sample Confusion Matrix:')
print(cm_outsample)

## 15. Full Classification Metrics

All five classification metrics are computed on the held-out test set. AUC is computed from raw probability scores and is threshold-independent, making it the primary metric for comparing discriminative ability across models.

These results correspond to the GRU row of **Table 7.1** in the dissertation.

In [ ]:
# Full classification metrics
accuracy    = accuracy_score(y_true, test_predicted_classes)
precision   = precision_score(y_true, test_predicted_classes, zero_division=0)
sensitivity = recall_score(y_true, test_predicted_classes, zero_division=0)
tn, fp, fn, tp = cm_outsample.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
auc = roc_auc_score(y_true, test_predictions)

print('=' * 55)
print('  GRU (VIX) Classification Results (Out-of-Sample)')
print('=' * 55)
print(f'  Accuracy:    {accuracy:.3f}')
print(f'  Precision:   {precision:.3f}')
print(f'  Sensitivity: {sensitivity:.3f}')
print(f'  Specificity: {specificity:.3f}')
print(f'  AUC:         {auc:.3f}')
print('=' * 55)
print()
print(f'Best hyperparameters: {best_params}')

## 16. Confusion Matrix Plot

The out-of-sample confusion matrix is plotted. This figure corresponds to **Figure 7.4** in the dissertation.

In [ ]:
# Confusion matrix plot
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm_outsample, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_title('GRU (VIX) Confusion Matrix (Out-of-Sample)', size=13)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred 0', 'Pred 1'], size=11)
ax.set_yticklabels(['Act 0',  'Act 1'],  size=11)
thresh = cm_outsample.max() / 2.
for i, j in itertools.product(range(cm_outsample.shape[0]), range(cm_outsample.shape[1])):
    ax.text(j, i, format(cm_outsample[i, j], 'd'),
            ha='center', va='center', size=12,
            color='white' if cm_outsample[i, j] > thresh else 'black')
ax.set_ylabel('Actual', size=11)
ax.set_xlabel('Predicted', size=11)
plt.tight_layout()
plt.savefig('NVDA_GRU_VIX_Confusion_Matrix.jpg', format='jpg', dpi=300, bbox_inches='tight')
plt.show()

## 17. ROC Curve

The ROC curve is plotted for the out-of-sample test period. The colour gradient along the curve represents the classification threshold, ranging from high (dark) to low (light). This figure corresponds to **Figure 7.3** in the dissertation. An AUC substantially above 0.5 confirms that the GRU's probability scores are genuinely informative about the direction of the next day's volatility.

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_true, test_predictions)

plt.figure(figsize=(10, 8))
sc = plt.scatter(fpr, tpr, c=thresholds, cmap='viridis', edgecolor='none', s=70)
plt.plot(fpr, tpr, color='black', lw=1)
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.gca().tick_params(axis='x', labelsize=15)
plt.gca().tick_params(axis='y', labelsize=15)
plt.xlabel('1 - Specificity', fontsize=20)
plt.ylabel('Sensitivity', fontsize=20)
cbar = plt.colorbar(sc)
cbar.set_label('Threshold', size=18)
cbar.ax.tick_params(labelsize=15)
plt.tight_layout()
plt.savefig('NVDA_GRU_VIX_ROC.jpg', format='jpg', dpi=300, bbox_inches='tight')
plt.show()

## 18. Dissertation Summary Table

The GRU classification results are printed in the format used in Table 7.1 of the dissertation for direct transcription.

In [ ]:
# Dissertation summary table
print(f'{"Model":<14} | {"Accuracy":>8} | {"Precision":>9} | {"Sensitivity":>11} | {"Specificity":>11} | {"AUC":>6}')
print('-' * 75)
print(f'{"GRU":<14} | {accuracy:>8.3f} | {precision:>9.3f} | {sensitivity:>11.3f} | {specificity:>11.3f} | {auc:>6.3f}')